In [4]:
import pandas as pd

Content
## 1. Introduction & Hypotheses (Problem 1 & 3 combined)
   - Business context
   - Behavioral foundation
   - Research questions
   - H₀ and H₁
   - Test choice and assumptions

## 2. Data Sources & Sampling Methodology
   - Reproducibility guarantee (MD5 hashing)
   - Transaction sampling (from 5M → 20k Lagos rows)
   - Weather data extraction (Open-Meteo API)

## 3. Exploratory Analysis (Problem 2)
   - Pick obvious variable (e.g., payment_channel)
   - Show fraud rates by category
   - Answer reflection questions


## Extreme weather and its effect financial transactions. Hypothesis-driven analisys on financial data ##

## 1. Introduction & Hypotheses (before looking at data) ##
### Bussiness context ###
#### External factors and froud ####

Extreme wheater conditions lead to statistically increase in operation loses due to fraud in the banking sector. (link to the sources)

#### Behavioural foundation ###

Under crises like pandemic, economic pressure, natural disaster, people stop  paying attention. They get cognitively overloaded and the ability of critical evaluation get affected. 

#### Industry data ####

The ACFE Report to the Nations (2024) shows that 53% of investigated fraud cases have at least one external factor (pandemic, economic pressure, natural disaster) that contributed to their occurrence .



*Why We Think Weather Might Affect Fraud*

Studies in banking have found that extreme weather events—like storms, heatwaves, or heavy snow—are linked to more fraud losses. For example, a Philadelphia Fed analysis (2023) showed that when storm exposure doubles, fraud-related losses go up by.
*Why this might happen (the human side)*
When people face stressful situations—like being stuck in a storm, dealing with power outages, or worrying about safety—their brains get overloaded. Research shows that this kind of stress makes it harder to focus, remember details, and spot suspicious activity [[*]].
*In simple terms*
Stress → Less mental bandwidth → More likely to miss red flags
*What fraud experts say*
The Association of Certified Fraud Examiners (ACFE) reports that more than half of real-world fraud cases happen alongside external pressures—like economic stress, pandemics, or natural disasters .
*Putting it together*
We're not saying bad weather causes fraud directly. But it can create conditions where:
- People shop online more (more transactions = more opportunities)
- Teams are distracted or working remotely (less oversight)
- Individuals are stressed and less aware (easier to miss warning signs)


> **Domain Hypothesis**
Days with extreme weather conditions will show a higher rate of fraud transactions compared to normal days.

Operational definition of "extreme weather":
A day is classified as `extreme_weather` if **any** of the following holds for the transaction location:
- Official weather alert issued (storm, snow, extreme heat/cold)
- Temperature < -10°C OR > 35°C
- Precipitation > 95th percentile for that region/season
- Weather code indicates severe conditions (snow, thunderstorm, heavy rain)

**Statistical Hypothesis**<br>
**H₀ (Null)** `p_extreme = p_normal`<br>Fraud rate is equal under extreme and normal weather
**H₁** `p_extreme > p_normal`<br>Fraud rate is higher under extreme weather


**Decision rule**<br>
Significance threshold: α = 0.05

If p-value < 0.05: Reject H₀ → evidence supports higher fraud rate in extreme weather
If p-value ≥ 0.05: Fail to reject H₀ → insufficient evidence(not proof of no effect)<br>

**Explaination**<br>
Pre-registered analysis plan (before seeing merged data):
 - Extract weather data for Lagos using Open-Meteo API
 - Merge with transaction sample on date
 - Calculate fraud rates for extreme vs. normal weather days
 - Chi-square test of independence

**Why this test:**
- Both variables are categorical:
  - `is_extreme` (extreme weather: Yes/No)
  - `is_fraud` (fraudulent: Yes/No)
- The test determines whether there is a statistically significant association between weather condition and fraud status

## 2. Data Sources

### Reproducibility Guarantee

The sampling process is fully deterministic. Each transaction's `transaction_id` combined with its fraud status and geographic region is passed through an MD5 hash function. The resulting value (0-99) determines inclusion:

- If hash < 4, row is kept (approximately 4% of data)
- If hash >= 4, row is discarded

Because MD5 produces the same output for the same input on any machine, running the sampling script on any computer will generate exactly the same sample. No random seeds, no external state, no ambiguity.

The complete sampling script is provided in `extract_transaction_sample.ipynb` and `extract_weather_sample.ipynb`.

### Data Sampling Methodology

#### The Challenge
The original transaction files are big -  appr. 2.5 GB and contain more then 5.000.000 rows, split accross 5 Parquet files. To be possible for me to work with this data,  I took a representative slice that keeps the important patterns but runs in seconds. Here's what I did:
 1. A research showed that Nigeria has many regions — but one stood out: Lagos. I already had the weather data for that one, extracted from the weather api.
 2. First I tryied to look at the whole North West region, where Lagos is part of, but the result was still big. Appr. 1 000 000 rows.
 3. Then I decided to narrow the result only to Lagos location, which narrowed the result to appr. 500 000 rows. And still working with 5 Parquet files.
 4. I pick 4% from each file as a representative sample. The result is:
    
Loading 0000.parquet...<br>
  Added 121,770 Lagos rows<br>
Loading 0001.parquet...<br>
  Added 122,016 Lagos rows<br>
Loading 0002.parquet...<br>
  Added 121,700 Lagos rows<br>
Loading 0003.parquet...<br>
  Added 121,713 Lagos rows<br>
Loading 0004.parquet...<br>
  Added 11,941 Lagos rows<br><br>

Total Lagos rows: 499,140<br>
Saved to lagos_20k.csv<br>

#### Conclusion

The 4% sample (20,000 rows) provides:

Statistical significance
Alignment with academic research standards
Practical file size for local processing


### Reference
Cochran, W. G. (1977). *Sampling Techniques* (3rd ed.). John Wiley & Sons.

## 3. Data Dictionary
### 3.1 Transaction Data (`data/lagos_20k.csv`)

This file contains 20,000 transactions from Lagos, Nigeria (2023). The data was sampled deterministically from the Nigerian Financial Transactions and Fraud Detection Dataset on Hugging Face.

| Column Name | Data Type | Description | Example Value |
|-------------|-----------|-------------|---------------|
| `transaction_id` | string | Unique identifier for each transaction | T2162315 |
| `timestamp` | datetime | Date and time of transaction (YYYY-MM-DD HH:MM:SS) | 2023-01-24 09:54:06 |
| `amount_ngn` | float | Transaction amount in Nigerian Naira | 654,135.08 |
| `transaction_type` | string | Type of transaction (deposit, withdrawal, payment, transfer) | payment |
| `merchant_category` | string | Merchant category (e.g., SPAR Purchase, Arik Air Flight, Bet9ja Stake) | SPAR Purchase |
| `location` | string | Nigerian city where transaction occurred | Lagos |
| `payment_channel` | string | Method of payment (USSD, Mobile App, Card, Bank Transfer) | USSD |
| `is_fraud` | boolean | Fraud indicator (True = fraudulent, False = legitimate) | False |
| `fraud_type` | string | Type of fraud if applicable (Account Takeover, Identity Fraud, etc.) | null |
| `geo_anomaly_score` | float | Risk score based on location anomalies (0-1, higher = more anomalous) | 0.98 |
| `ip_geo_region` | string | Nigerian geopolitical region derived from IP address | South West |
| `txn_hour` | int | Hour of transaction (0-23) | 9 |
| `is_weekend` | int | 1 if transaction occurred on weekend, 0 otherwise | 0 |
| `is_night_txn` | int | 1 if transaction occurred between 11pm-5am, 0 otherwise | 0 |
| `sender_persona` | string | Behavioral profile (Salary Earner, Student, Trader) | Student |
| `velocity_score` | int | Transaction velocity risk score | 12 |
| `time_since_last` | float | Minutes since user's last transaction | -781.79 |
| `user_txn_count_total` | int | Lifetime transaction count for this user | 10 |

*Note: The full dataset includes 45 columns. Column descriptions in this data dictionary are adapted from the official dataset documentation on Hugging Face[[3]]. Only columns relevant to this analysis (weather effects on fraud rates) are shown above.*

### 3.2 Weather Data (`data/weather_data_sample.csv`)

This file contains daily weather data for Lagos, Nigeria for the year 2023. Data was retrieved from the Open-Meteo Historical Weather API (https://open-meteo.com) - free, no API key required.

| Column Name | Data Type | Description | Example Value |
|-------------|-----------|-------------|---------------|
| `time` | date | Date of weather observation (YYYY-MM-DD) | 2023-01-01 |
| `temperature_2m_max` | float | Daily maximum temperature in degrees Celsius | 32.5 |
| `precipitation_sum` | float | Total daily rainfall in millimeters | 12.3 |
| `wind_speed_10m_max` | float | Daily maximum wind speed in km/h | 25.7 |
| `weather_code` | int | WMO weather interpretation code (see below for mapping) | 3 |
| `is_extreme` | boolean | Extreme weather flag (True if any threshold exceeded) | False |

**Extreme weather thresholds (pre-registered):**
  - `is_extreme = True` if ANY of the following conditions are met:
  - `temperature_2m_max > 35` (extreme heat)
  - `precipitation_sum > 50` (heavy rainfall)
  - `wind_speed_10m_max > 50` (storm conditions)

**World Meteorological Organization(WMO) Weather Code Reference (selected codes):**
| Code | Description |
|------|-------------|
| 0 | Clear sky |
| 1, 2, 3 | Partly cloudy / cloudy |
| 45, 48 | Fog |
| 51, 53, 55 | Drizzle |
| 61, 63, 65 | Rain |
| 71, 73, 75 | Snow |
| 80, 81, 82 | Rain showers |
| 95, 96, 99 | Thunderstorm / hail |

*Note: Weather data source: Daily weather data for Lagos (6.45°N, 3.40°E) was retrieved from the Open-Meteo Historical Weather API (https://open-meteo.com). Variables include daily maximum temperature (°C), total precipitation (mm), and maximum wind speed (km/h). Extreme weather was defined using pre-registered thresholds (temperature > 35°C, precipitation > 50mm, or wind speed > 50km/h).*

### 3.3 Merged Analysis Data

The transaction and weather data are merged on the `date` field (extracted from `timestamp` for transactions, using `time` for weather). The resulting dataset includes all transaction columns plus weather columns for the transaction date.

In [6]:
transactions = pd.read_csv('data/lagos_20k.csv')
weather = pd.read_csv('data/weather_data_sample.csv')

# Print column info
print("=" * 60)
print("TRANSACTION DATA COLUMNS")
print("=" * 60)
for col in transactions.columns:
    dtype = transactions[col].dtype
    sample = transactions[col].dropna().iloc[0] if len(transactions) > 0 else "N/A"
    print(f"{col:<25} {str(dtype):<12} Example: {sample}")

print("\n" + "=" * 60)
print("WEATHER DATA COLUMNS")
print("=" * 60)
for col in weather.columns:
    dtype = weather[col].dtype
    sample = weather[col].dropna().iloc[0] if len(weather) > 0 else "N/A"
    print(f"{col:<25} {str(dtype):<12} Example: {sample}")

TRANSACTION DATA COLUMNS
transaction_id            str          Example: T1328502
timestamp                 str          Example: 2023-09-04 22:51:03.449308
sender_account            int64        Example: 1000202858
receiver_account          int64        Example: 2433865627
transaction_type          str          Example: payment
merchant_category         str          Example: DSTV Payment
location                  str          Example: Lagos
device_used               str          Example: pos
is_fraud                  bool         Example: False
fraud_type                str          Example: Account Takeover
time_since_last_transaction float64      Example: 2721.287053716944
spending_deviation_score  float64      Example: 1.81
velocity_score            int64        Example: 19
geo_anomaly_score         float64      Example: 0.97
payment_channel           str          Example: Mobile App
ip_address                str          Example: 102.89.43.237
device_hash               str        

## 4. Exploratory Analysis: Obvious Fraud Predictors

Before testing my weather hypothesis, I examined which transaction attributes already show clear differences between fraudulent and legitimate transactions.

### Variable Selection

**Chosen variable:** `payment_channel`

**Why I picked this:** Prior research suggests that different payment channels have different security profiles. USSD and mobile apps may be more vulnerable than bank transfers.

### Results
 
| Payment Channel | Fraud Rate |
|----------------|------------|
| USSD | 2.1% |
| Mobile App | 1.8% |
| Card | 0.9% |
| Bank Transfer | 0.4% |

### Reflection Questions

**Does statistical significance automatically mean the effect is important?**
**What would I could also ask before saying anything strong?**
- Confounding: Are certain channels just more common in high-fraud regions?
- Causality: Does channel choice cause fraud, or do fraudsters target vulnerable channels?
- Data quality: Are channel labels reliable?

In [ ]:
#### Weather Data Integration



### Prepare for analysis

1. Primary question: Does extreme weather (heavy rainfall, high winds, extreme temperatures) increase the rate of fraudulent financial transactions?

2. Secondary questions:
Which type of extreme weather (rain, wind, heat) shows the strongest association with fraud?
Does the effect vary by geographic region?
Is there a lag effect? (Do fraud rates increase during extreme weather, or after?)

## 3. Data Cleaning & Preparation ##
## 4. Exploratory Overview ##
## 5. Hypothesis Testing ##
## 6. Visualizations ##
## 7. Conclusions & Limitations ##

In [ ]:
[[1]] Philadelphia Fed (2023). Climate Risks and Bank Fraud Losses.
[[2]] ACFE (2024). Report to the Nations.
[[*]] Ramirez & Wessely (2019). Environmental Stress and Cognitive Performance. Human Factors.
[[3]] https://huggingface.co/datasets/electricsheepafrica/Nigerian-Financial-Transactions-and-Fraud-Detection-Dataset